# 03 — Testing a Claim

Distributions unit, chapter 3. Setup, one line to change, what you should
see, going further.


## Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binom

rng = np.random.default_rng(seed=11)


## Part A — p-values, exactly as on the calculator

The three worked examples from the chapter.


In [ ]:
print("retailer:  p(X >= 3), n=20, p=0.04  ->", round(1 - binom.cdf(2, 20, 0.04), 4))
print("seeds:     p(X <= 40), n=50, p=0.90 ->", round(binom.cdf(40, 50, 0.90), 4))
print("coin, 15H: p(X >= 15), n=20, p=0.50 ->", round(1 - binom.cdf(14, 20, 0.5), 4))
print("           two-sided               ->", round(2 * (1 - binom.cdf(14, 20, 0.5)), 4))
print("coin, 14H: two-sided               ->", round(2 * (1 - binom.cdf(13, 20, 0.5)), 4))


**What you should see.** 0.0439, 0.0245, 0.0207, 0.0414 and 0.1153.

Look at the last two. One extra head takes the two-sided p-value from 0.115
to 0.041 — across the 0.05 line. That is the whole argument against reporting
"significant" instead of the number itself.


### The rejection region by trial

The chapter finds the critical value by trying a few k. So does this:


In [ ]:
n, p0, alpha = 20, 0.5, 0.05

for k in range(12, 18):
    tail = 1 - binom.cdf(k - 1, n, p0)
    mark = "  <-- reject from here" if tail <= alpha else ""
    print(f"k = {k}:  p(X >= k) = {tail:.4f}{mark}")


## Part B — the test is stricter than it was asked to be

The chapter claims this test rejects a fair coin only 2.1% of the time, not
5%, because X takes whole-number values and 0.05 is not on the available
list. Rather than trust that, run the test two hundred thousand times.

### THE ONE LINE TO CHANGE

`TRUE_P` is the coin's real bias. Leave it at 0.5 to measure the Type I
error rate; set it to 0.7 to measure the power.


In [ ]:
TRUE_P = 0.5          # <-- change this one number

REPEATS = 200000
heads = rng.binomial(20, TRUE_P, size=REPEATS)
reject = (heads >= 15)

print("true p       =", TRUE_P)
print("rejected     =", round(reject.mean(), 4))
print("exact answer =", round(1 - binom.cdf(14, 20, TRUE_P), 4))


**What you should see.** At `TRUE_P = 0.5`, about 0.021 — the real
significance level, well under the 5% we asked for. At 0.7, about 0.42:
a badly biased coin escapes this test more often than it is caught.

Twenty tosses is simply not much evidence.


## Going further


### G1 — the whole power curve

Instead of one value of the true p, sweep all of them.


In [ ]:
ps = np.linspace(0.3, 0.95, 100)
power = 1 - binom.cdf(14, 20, ps)

plt.plot(ps, power)
plt.axhline(0.05, color="grey", linestyle=":", label="alpha = 0.05")
plt.axvline(0.5, color="crimson", linestyle="--", label="H0: p = 0.5")
plt.xlabel("true value of p")
plt.ylabel("probability of rejecting H0")
plt.legend()
plt.show()


Read the curve at p = 0.5 and you get the Type I error rate. Read it
anywhere else and you get the power against that alternative. A good test
has a curve that is low at 0.5 and rises steeply — this one rises lazily,
which is what a sample of 20 buys you.

Change the 20 and the 14 to 200 and 114 and look again.


### G2 — the multiple testing problem

The chapter claims that testing twenty *false* hypotheses at the 5% level
produces at least one significant result about 64% of the time. This runs
the experiment: twenty fair coins, each tested for bias, none of them
actually biased.


In [ ]:
STUDIES = 20000

heads = rng.binomial(20, 0.5, size=(STUDIES, 20))
sig = (heads >= 15) | (heads <= 5)
any_sig = sig.any(axis=1)

print("at least one significant result:", round(any_sig.mean(), 4))
print("average number of them:         ", round(sig.sum(axis=1).mean(), 4))


**What you should see.** Roughly 0.57 and 0.83 — a little below the 0.64
and 1.0 in the notes, because the real per-test rejection rate is 0.041
two-sided rather than the nominal 0.05. The point survives intact: a
researcher who runs twenty tests and reports the one that worked has
probably found nothing.

This is Littlewood's law from last year's paradoxes chapter, in a lab coat.


### G3 — your turn

Pick a claim you could actually test with 30 trials — a friend's ability to
guess a coin, the proportion of red sweets in a bag, anything. Write down
H0, H1 and alpha *before* collecting anything. Find the rejection region
here. Then collect the data and report the p-value, whatever it says.

Deciding the rejection region first is the part that makes it a test rather
than a story.
